# Inkwell GPU experiments
Enable a CUDA GPU runtime before running. Upload the prepared `inkwell-gpu-input.zip`. This bundle contains research datasets; keep checkpoints and data within the course. Modern full-label evaluation requires completed human reviews.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
workspace = Path("/content/inkwell")
workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    z.extractall(workspace)
%cd /content/inkwell


In [ ]:
%pip install numpy==1.26.4 scikit-learn==1.7.2 joblib==1.5.3 transformers==4.57.6
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime first"
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
!python -m unittest discover -s tests -v


## Run the staged matrix
This trains both backbones on identical frozen splits, compares C3PA weights 0.1 and 1.0 for the better validation backbone, and repeats shortlisted configurations across seeds 42, 13, and 7. Validation selects the configuration. Only the predetermined seed-42 representative is evaluated on test. Completed runs are reused; partially written runs require a new output path.


In [ ]:
import os
os.environ["HF_HOME"] = "/content/inkwell/.model-cache"
!python -m scripts.run_gpu_experiments --out runs/gpu --epochs 10


In [ ]:
import json
from pathlib import Path
selection = json.loads(Path("runs/gpu/selection.json").read_text())
print(json.dumps(selection, indent=2))


## Save checkpoints before the runtime ends
The archive includes models and tokenizers and can be large. Download it into course storage before ending the session. Never publish research checkpoints automatically. Domain-adaptive pretraining is deferred until the supervised comparisons and modern reviewed benchmark are reliable.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/inkwell-gpu-results", "zip", "/content/inkwell/runs/gpu")
files.download(archive)
